In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import month_plot, quarter_plot, plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import seaborn as sns

import shap

/home/albertouskas/Desktop/Data Playground/traffic_flow_prediction_forecasting_and_analysis/.venv-tsfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

data_loc = r"~/Desktop/Data Playground/Traffic Flow Forecasting and Analysis"
data_name="traffic_sensor_data.csv"
#df = pd.read_csv(f"{data_loc}/{data_name}") # Linux Option
df = pd.read_csv(f"{data_name}") # Windows Option

## Feature Engineering

In [3]:
def feature_engineering(
    df,
    target_column='vehicle_count',
    lag_steps=(1, 2),
    rolling_window=6,
    keep_timestamp=False,
    drop_na=True,
):
    modern_df = df.copy()
    modern_df['timestamp'] = pd.to_datetime(modern_df['timestamp'])
    modern_df = modern_df.sort_values(['sensor_id', 'timestamp']).reset_index(drop=True)

    # Calendar features are known in advance, so they are valid for forecasting
    modern_df['month'] = modern_df['timestamp'].dt.month
    modern_df['hour'] = modern_df['timestamp'].dt.hour
    modern_df['minute'] = modern_df['timestamp'].dt.minute
    modern_df['day_of_week'] = modern_df['timestamp'].dt.dayofweek
    modern_df['slot_30m'] = modern_df['hour'] * 2 + (modern_df['minute'] // 30)
    modern_df['slot_30m_sin'] = np.sin(2 * np.pi * modern_df['slot_30m'] / 48)
    modern_df['slot_30m_cos'] = np.cos(2 * np.pi * modern_df['slot_30m'] / 48)
    modern_df['day_of_week_sin'] = np.sin(2 * np.pi * modern_df['day_of_week'] / 7)
    modern_df['day_of_week_cos'] = np.cos(2 * np.pi * modern_df['day_of_week'] / 7)
    modern_df['month_sin'] = np.sin(2 * np.pi * (modern_df['month'] - 1) / 12)
    modern_df['month_cos'] = np.cos(2 * np.pi * (modern_df['month'] - 1) / 12)

    # Past-only target history per sensor; shift(1) keeps the current value out of the rolling mean
    grouped_target = modern_df.groupby('sensor_id')[target_column]
    for step in lag_steps:
        modern_df[f'{target_column}_lag_{step}'] = grouped_target.shift(step)
    modern_df[f'{target_column}_roll_mean_{rolling_window}'] = grouped_target.transform(
        lambda s: s.shift(1).rolling(rolling_window, min_periods=rolling_window).mean()
    )

    # Sensor as one integer code; sorted so each sensor keeps the same code in every split
    sensor_codes = {sensor: code for code, sensor in enumerate(sorted(modern_df['sensor_id'].unique()))}
    modern_df['sensor_id'] = modern_df['sensor_id'].map(sensor_codes).astype('int16')

    # Same-slot measurements are unknown at forecast time, and vehicle-type counts sum to the target
    columns_to_drop = [
        'location', 'direction', 'average_speed_kmh', 'occupancy_rate', 'congestion_level',
        'car_count', 'motorcycle_count', 'truck_count', 'bus_count',
        'month', 'hour', 'minute', 'day_of_week', 'slot_30m',
    ]
    if not keep_timestamp:
        columns_to_drop.append('timestamp')
    modern_df = modern_df.drop(columns=columns_to_drop)

    if drop_na:
        history_cols = [
            c for c in modern_df.columns
            if c.startswith(f'{target_column}_lag_') or c.startswith(f'{target_column}_roll_mean_')
        ]
        modern_df = modern_df.dropna(subset=history_cols).reset_index(drop=True)

    return modern_df


def classical_feature_engineering(df, target_column='vehicle_count', **kwargs):
    # Same columns as feature_engineering, indexed by time for the TSFM pipeline
    feats = feature_engineering(df, target_column=target_column, keep_timestamp=True, **kwargs)
    return feats.rename(columns={'timestamp': 'ts'}).set_index('ts')


## Spliting Datasets

In [5]:
# Chronological split on unique timestamps, so each timestamp's sensors stay in the same split
TRAIN_FRACTION = 0.8
VALIDATION_SET = False  # False, or the share of timestamps after train used for validation

def split_dataset(df, classical_full, modern_full, train_fraction=TRAIN_FRACTION, validation_set=VALIDATION_SET):
    timestamps = pd.to_datetime(df['timestamp']).drop_duplicates().sort_values().reset_index(drop=True)
    n = len(timestamps)

    validation_share = 0.0 if validation_set is False else float(validation_set)
    if not 0 <= validation_share <= 1 - train_fraction:
        raise ValueError('validation_set must be False or a share that fits in the remaining timestamps')

    train_end = int(n * train_fraction)
    val_end = train_end + int(n * validation_share)
    train_cut = timestamps.iloc[train_end - 1]
    val_cut = timestamps.iloc[val_end - 1] if val_end > train_end else train_cut

    classical_times = pd.DatetimeIndex(classical_full.index)
    modern_times = pd.DatetimeIndex(modern_full['timestamp'])
    classical_labels = np.select([classical_times <= train_cut, classical_times <= val_cut], ['train', 'validation'], 'test')
    modern_labels = np.select([modern_times <= train_cut, modern_times <= val_cut], ['train', 'validation'], 'test')

    splits = {}
    for name in ['train', 'validation', 'test']:
        splits[name] = {
            'classical': classical_full[classical_labels == name],
            'modern': modern_full[modern_labels == name].reset_index(drop=True),
        }

    print(
        f'Train cutoff: {train_cut} | Validation cutoff: {val_cut} | '
        f"Train timestamps: {(timestamps <= train_cut).sum():,} | "
        f"Validation timestamps: {((timestamps > train_cut) & (timestamps <= val_cut)).sum():,} | "
        f"Test timestamps: {(timestamps > val_cut).sum():,}"
    )
    return splits


# Features are built on the full series first so test rows keep their history from train
classical_full = classical_feature_engineering(df, target_column='vehicle_count')
modern_full = feature_engineering(df, target_column='vehicle_count', keep_timestamp=True)

splits = split_dataset(df, classical_full, modern_full)
classical_train_df = splits['train']['classical']
classical_validation_df = splits['validation']['classical']
classical_test_df = splits['test']['classical']
modern_train_df = splits['train']['modern']
modern_validation_df = splits['validation']['modern']
modern_test_df = splits['test']['modern']

Train cutoff: 2024-03-13 18:30:00 | Validation cutoff: 2024-03-13 18:30:00 | Train timestamps: 3,494 | Validation timestamps: 0 | Test timestamps: 874


## ML SHAP Analysis

## TFMs SHAP Analysis